In [2]:
import sklearn
import seaborn as sns
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint
import math
pd.__version__

'2.3.3'

In [4]:
import os
import mlflow

os.environ['AWS_PROFILE'] = '786875389232'

TRACKING_URI = 'ec2-51-21-167-235.eu-north-1.compute.amazonaws.com'
mlflow.set_tracking_uri(f'http://{TRACKING_URI}:5000')
mlflow.set_experiment('nyc-taxi-experiment')

<Experiment: artifact_location='s3://mlflow-artifacts-remote-786875389232/1', creation_time=1794424515109, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794424515109, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [5]:
def read_data(file):
    df = pd.read_parquet(file)
    print(f"Number of columns for {file}: {len(df.columns)}")
    # print(df.dtypes)
    #convert to datetime64[ns] to int64
    converted_start_end_df = df.astype({"tpep_pickup_datetime": "int", "tpep_dropoff_datetime": "int"})
    df['duration'] = (converted_start_end_df['tpep_dropoff_datetime'] - converted_start_end_df['tpep_pickup_datetime']) * 1.6666666666667e-8
    # print(df.duration.describe())
    print(f"min: {df.duration.min()}")
    print(f"max: {df.duration.max()}")
    df_b060 = df[(df.duration >= 1) & (df.duration <= 60)]
    # Q:What fraction of the records left after you dropped the outliers?
    fraction = len(df_b060) / len(df)
    print(f"fraction: {fraction*100:.2f}")
    return df_b060[["PULocationID", "DOLocationID", "duration"]]

In [8]:
train_df = read_data("./data/yellow_tripdata_2023-01.parquet")
print("done reading data")

Number of columns for ./data/yellow_tripdata_2023-01.parquet: 19
min: -29.200000000000585
max: 10029.183333333534
fraction: 98.12
done reading data


In [6]:
val_df = read_data("./data/yellow_tripdata_2023-02.parquet")

Number of columns for ./data/yellow_tripdata_2023-02.parquet: 19
min: -43.61666666666754
max: 7053.616666666809
fraction: 98.01


In [21]:
# sns.distplot(train_df.duration, label="train")
# sns.distplot(val_df.duration, label="val")
# plt.legend()

In [9]:
categorical = ["PULocationID", "DOLocationID"]
numerical = ['trip_distance']

vectorizer = DictVectorizer()

train_dicts = train_df[categorical].astype(str).to_dict(orient='records')
pprint(train_dicts[:5])
X_train = vectorizer.fit_transform(train_dicts)
del train_dicts

[{'DOLocationID': '141', 'PULocationID': '161'},
 {'DOLocationID': '237', 'PULocationID': '43'},
 {'DOLocationID': '238', 'PULocationID': '48'},
 {'DOLocationID': '7', 'PULocationID': '138'},
 {'DOLocationID': '79', 'PULocationID': '107'}]


In [10]:
val_dicts = val_df[categorical].astype(str).to_dict(orient='records')
pprint(val_dicts[:5])
X_val = vectorizer.transform(val_dicts)
del val_dicts

[{'DOLocationID': '163', 'PULocationID': '142'},
 {'DOLocationID': '26', 'PULocationID': '132'},
 {'DOLocationID': '145', 'PULocationID': '161'},
 {'DOLocationID': '236', 'PULocationID': '148'},
 {'DOLocationID': '244', 'PULocationID': '137'}]


In [11]:
print(X_val.shape)

(2855879, 515)


In [12]:
target = "duration"
y_train = train_df[target].values
y_val = val_df[target].values

train = xgb.DMatrix(X_train, label=y_train)
validation = xgb.DMatrix(X_val, label=y_val)

In [13]:
del train_df
del val_df

In [ ]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("developer", "sergio")
        mlflow.log_param("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=50,
            evals=[(validation, "validation")],
            early_stopping_rounds=10
        )
        y_pred = booster.predict(validation)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
    return {"loss": rmse, "status": STATUS_OK}

search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)



  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [19:19:35] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:6.99094                           
[1]	validation-rmse:5.91612                           
[2]	validation-rmse:5.69654                           
[3]	validation-rmse:5.45133                           
[4]	validation-rmse:5.35719                           
[5]	validation-rmse:5.30017                           
[6]	validation-rmse:5.26974                           
[7]	validation-rmse:5.24492                           
[8]	validation-rmse:5.24026                           
[9]	validation-rmse:5.23068                           
[10]	validation-rmse:5.22759                          
[11]	validation-rmse:5.22581                          
[12]	validation-rmse:5.22172                          
[13]	validation-rmse:5.22122                          
[14]	validation-rmse:5.22092                          
[15]	validation-rmse:5.21979                          
[16]	validation-rmse:5.21920                          
[17]	validation-rmse:5.21859                          
[18]	valid

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [19:20:45] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:9.54876                                                    
[1]	validation-rmse:9.10552                                                    
[2]	validation-rmse:8.72957                                                    
[3]	validation-rmse:8.33306                                                    
[4]	validation-rmse:8.07412                                                    
[5]	validation-rmse:7.66486                                                    
[6]	validation-rmse:7.32481                                                    
[7]	validation-rmse:7.04982                                                    
[8]	validation-rmse:6.89277                                                    
[9]	validation-rmse:6.78428                                                    
[10]	validation-rmse:6.57100                                                   
[11]	validation-rmse:6.41488                                                   
[12]	validation-rmse:6.34734            

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [19:21:52] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



... (5 outputs left out)

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [19:23:44] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:9.62693                                                    
[1]	validation-rmse:9.24927                                                    
[2]	validation-rmse:8.92270                                                    
[3]	validation-rmse:8.60924                                                    
[4]	validation-rmse:8.37245                                                    
[5]	validation-rmse:8.03840                                                    
[6]	validation-rmse:7.85557                                                    
[7]	validation-rmse:7.58688                                                    
[8]	validation-rmse:7.37084                                                    
[9]	validation-rmse:7.25915                                                    
[10]	validation-rmse:7.08364                                                   
[11]	validation-rmse:7.00246                                                   
[12]	validation-rmse:6.92651            

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [19:24:33] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:9.78476                                                    
[1]	validation-rmse:9.52802                                                    
[2]	validation-rmse:9.29223                                                    
[3]	validation-rmse:9.07666                                                    
[4]	validation-rmse:8.87981                                                    
[5]	validation-rmse:8.69554                                                    
[6]	validation-rmse:8.51505                                                    
[7]	validation-rmse:8.36113                                                    
[8]	validation-rmse:8.22034                                                    
[9]	validation-rmse:8.00809                                                    
[10]	validation-rmse:7.81580                                                   
[11]	validation-rmse:7.70521                                                   
[12]	validation-rmse:7.53010            

In [29]:
print("Best hyperparameters:", best_result)

Best hyperparameters: {'learning_rate': np.float64(0.20844413031306103), 'max_depth': np.float64(99.0), 'min_child_weight': np.float64(1.5432280770679585), 'reg_alpha': np.float64(0.04953891913131428), 'reg_lambda': np.float64(0.017060826809340544)}


In [ ]:
import numpy as np

with mlflow.start_run():
    mlflow.set_tag("developer", "sergio")
    mlflow.log_param("model", "xgboost")
    mlflow.log_params(best_result)
    best_result = {'learning_rate': np.float64(0.20844413031306103), 
                   'max_depth': np.int64(99.0), 
                   'min_child_weight': np.float64(1.5432280770679585), 
                   'reg_alpha': np.float64(0.04953891913131428), 
                   'reg_lambda': np.float64(0.017060826809340544)}

    booster = xgb.train(
        params=best_result,
        dtrain=train,
        num_boost_round=50,
        evals=[(validation, "validation")],
        early_stopping_rounds=10
    )
    y_pred = booster.predict(validation)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

[0]	validation-rmse:8.91748
[1]	validation-rmse:8.15197
[2]	validation-rmse:7.63616
[3]	validation-rmse:6.90823
[4]	validation-rmse:6.38184
[5]	validation-rmse:6.02862
[6]	validation-rmse:5.83786
[7]	validation-rmse:5.75607
[8]	validation-rmse:5.68995
[9]	validation-rmse:5.54272
[10]	validation-rmse:5.51333
[11]	validation-rmse:5.42146
[12]	validation-rmse:5.40578
[13]	validation-rmse:5.34817
[14]	validation-rmse:5.33868
[15]	validation-rmse:5.30203
[16]	validation-rmse:5.29639
[17]	validation-rmse:5.28970
[18]	validation-rmse:5.26405
[19]	validation-rmse:5.26045
[20]	validation-rmse:5.24981
[21]	validation-rmse:5.24732
[22]	validation-rmse:5.24423
[23]	validation-rmse:5.23168
[24]	validation-rmse:5.23040
[25]	validation-rmse:5.22892
[26]	validation-rmse:5.22245
[27]	validation-rmse:5.22168
[28]	validation-rmse:5.21730
[29]	validation-rmse:5.21640
[30]	validation-rmse:5.21519
[31]	validation-rmse:5.21362
[32]	validation-rmse:5.21210
[33]	validation-rmse:5.20873
[34]	validation-rmse:5.2